In [5]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import networkx as nx
from pathlib import Path

from neural_reconstruction.core.topology import TopologyBuilder
from neural_reconstruction.core.pathfinding import PathFinder
from neural_reconstruction.algorithms.fragment_linking.utils import compute_vector_angle,is_direction_too_similar
from neural_reconstruction.core.preprocessing import dilate_epidermis_vertically

import skimage as ski
# from skimage.measure import label
from scipy.spatial import KDTree


In [6]:
BASE_PATH = Path('/home/pony/projects/ienf_q/data_0331')
IMAGE_ID = 'S222-2_a'



In [7]:
image      = cv2.imread(f'{BASE_PATH}/{IMAGE_ID}/image.png',       cv2.IMREAD_COLOR_RGB)[:, :, 1]  # 只取綠色通道
mask       = cv2.imread(f'{BASE_PATH}/{IMAGE_ID}/mask.png',        cv2.IMREAD_GRAYSCALE)
annotation = cv2.imread(f'{BASE_PATH}/{IMAGE_ID}/weka.png',        cv2.IMREAD_GRAYSCALE)
label      = cv2.imread(f'{BASE_PATH}/{IMAGE_ID}/label.png',       cv2.IMREAD_GRAYSCALE)


In [8]:

roi_mask = dilate_epidermis_vertically(mask, offset_px=50)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (51, 51))
background = cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel)
image = cv2.subtract(image, background)

roi_image = cv2.bitwise_and(image, image, mask=roi_mask)
roi_annotation = cv2.bitwise_and(annotation, annotation, mask=roi_mask)

# apply opening to roi_annotation to remove small noise
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
# roi_annotation = cv2.morphologyEx(roi_annotation, cv2.MORPH_OPEN, kernel)

# apply closing to roi_annotation to fill small holes
roi_annotation = cv2.morphologyEx(roi_annotation, cv2.MORPH_CLOSE, kernel, iterations=3)
roi_annotation[roi_annotation > 0] = 255


gt_builder = TopologyBuilder()
roi_label = cv2.bitwise_and(label, label, mask=roi_mask)
roi_label = cv2.morphologyEx(roi_label, cv2.MORPH_CLOSE, kernel, iterations=3)
gt_graph = gt_builder.build_seed_graph(roi_label)


In [9]:

annotation_component = np.asarray(ski.measure.label(roi_annotation, connectivity=2))

# 2. 骨架化 + 種子切分
topology_builder = TopologyBuilder(segment_length=3.0)
global_graph = topology_builder.build_seed_graph(roi_annotation, roi_image)

# 從 labeled image 補上 component_id
for node in global_graph.nodes():
    y, x = node
    global_graph.nodes[node]["component_id"] = int(annotation_component[int(y), int(x)])

# 元件內邊設低成本
# for u, v, data in global_graph.edges(data=True):
#     data["weight"] = 1e-5
# 3. 元件間路徑查找
cost_map = ((255 - roi_image.astype(np.float64)) / 255) ** 2


In [10]:
path_finder = PathFinder(cost_map)
topology_points = np.array(list(global_graph.nodes()))

kdtree = KDTree(topology_points)
seed_map = np.zeros_like(cost_map, dtype=bool)
for p in topology_points:
    seed_map[p[0], p[1]] = True

path_finder = PathFinder(cost_map)
topology_points = np.array(list(global_graph.nodes()))

path_lookup = path_finder.find_paths_from_seeds(
    topology_points=topology_points,
    kdtree=kdtree,
    search_radius=100,
    seed_map=seed_map,
    label_img=annotation_component,
)

In [11]:
from scipy.interpolate import interp1d

label_component = np.asarray(ski.measure.label(roi_label, connectivity=2))

valid_path = []
invalid_path = []
seen = set()
for (source, target), (path, cost) in path_lookup.items():
    key = (min(source, target), max(source, target))
    if key in seen:
        continue
    seen.add(key)

    path_arr = np.array(path)
    middle = path_arr[1:-1]

    mostly_on_annotation = (
        len(middle) > 0 and
        np.mean(roi_label[middle[:, 0], middle[:, 1]] > 0) >= 0.7
    )
    same_component = (
        label_component[int(source[0]), int(source[1])] ==
        label_component[int(target[0]), int(target[1])]
    )

    if mostly_on_annotation and same_component:
        valid_path.append((source, target, path, cost))
    else:
        invalid_path.append((source, target, path, cost))

print(f"Valid paths:   {len(valid_path)}")
print(f"Invalid paths: {len(invalid_path)}")

Valid paths:   1225
Invalid paths: 6844


In [12]:
# Visualize valid and invalid paths drawn on the image
fig, axes = plt.subplots(2, 1, figsize=(128, 64), constrained_layout=True)

for ax, path_list, color, title in [
    (axes[0], valid_path,   (0, 255, 0),   f'Valid Paths (n={len(valid_path)})'),
    (axes[1], invalid_path, (255, 80, 80),  f'Invalid Paths (n={len(invalid_path)})'),
]:
    viz = cv2.cvtColor(roi_image, cv2.COLOR_GRAY2RGB)
    viz[roi_label > 0] = [0,0,255]
    for source, target, path, cost in path_list:
        for i in range(len(path) - 1):
            y0, x0 = path[i]
            y1, x1 = path[i + 1]
            cv2.line(viz, (x0, y0), (x1, y1), color, 1)
    ax.imshow(viz)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Valid vs Invalid Path Visualization on Image', fontsize=14)
plt.show()

In [13]:
from scipy.interpolate import interp1d
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_DIR = Path('/home/pony/projects/ienf_q/data_0331')
sample_dirs = sorted([p for p in BASE_DIR.iterdir() if p.is_dir()])

def process_sample(sample_path):
    name = sample_path.name
    try:
        img   = cv2.imread(str(sample_path / 'image.png'),  cv2.IMREAD_COLOR_RGB)[:, :, 1]
        msk   = cv2.imread(str(sample_path / 'mask.png'),   cv2.IMREAD_GRAYSCALE)
        annot = cv2.imread(str(sample_path / 'weka.png'),   cv2.IMREAD_GRAYSCALE)
        lbl   = cv2.imread(str(sample_path / 'label.png'),  cv2.IMREAD_GRAYSCALE)

        roi_msk = dilate_epidermis_vertically(msk, offset_px=50)
        kernel_bg = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (51, 51))
        bg = cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel_bg)
        img = cv2.subtract(img, bg)

        roi_img   = cv2.bitwise_and(img,   img,   mask=roi_msk)
        roi_annot = cv2.bitwise_and(annot, annot, mask=roi_msk)

        kernel_cl = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        roi_annot = cv2.morphologyEx(roi_annot, cv2.MORPH_CLOSE, kernel_cl, iterations=3)
        roi_annot[roi_annot > 0] = 255

        annot_component = np.asarray(ski.measure.label(roi_annot, connectivity=2))

        topo = TopologyBuilder(segment_length=3.0)
        g = topo.build_seed_graph(roi_annot, roi_img)
        for node in g.nodes():
            y, x = node
            g.nodes[node]['component_id'] = int(annot_component[int(y), int(x)])

        c_map = ((255 - roi_img.astype(np.float64)) / 255) ** 2
        tpts  = np.array(list(g.nodes()))
        kd    = KDTree(tpts)
        s_map = np.zeros_like(c_map, dtype=bool)
        for p in tpts:
            s_map[int(p[0]), int(p[1])] = True

        pf = PathFinder(c_map)
        pl = pf.find_paths_from_seeds(
            topology_points=tpts, kdtree=kd,
            search_radius=50, seed_map=s_map, label_img=annot_component,
        )
        lbl_comopnent = np.asarray(ski.measure.label(lbl, connectivity=2))
        v, iv, seen = [], [], set()
        for (src, tgt), (path, cost) in pl.items():
            key = (min(src, tgt), max(src, tgt))
            if key in seen:
                continue
            seen.add(key)
            path_arr = np.array(path)
            mid = path_arr[1:-1]

            mostly_on_annotation = (
                len(mid) > 0 and
                np.mean(roi_annot[mid[:, 0], mid[:, 1]] > 0) >= 0.8
            )
            same_component = (
                lbl_comopnent[int(src[0]), int(src[1])] == lbl_comopnent[int(tgt[0]), int(tgt[1])]
            )

            if mostly_on_annotation and same_component:
                v.append((src, tgt, path, cost))
            else:
                iv.append((src, tgt, path, cost))
        return name, v, iv, roi_img
    except Exception as e:
        print(f"  [skip] {name}: {e}")
        return name, [], [], None

# ── Parallel execution ────────────────────────────────────────────────────────
N_WORKERS = min(24, len(sample_dirs))
results_map = {}

with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = {pool.submit(process_sample, sd): sd.name for sd in sample_dirs}
    for fut in as_completed(futures):
        name, v, iv, roi_img = fut.result()
        results_map[name] = (v, iv, roi_img)
        print(f"  {name}: valid={len(v)}, invalid={len(iv)}")

# Restore sorted order
sample_results = [(name, *results_map[name]) for name in [sd.name for sd in sample_dirs] if name in results_map]

all_valid   = [(src, tgt, path, cost, roi_img) for _, v, _,  roi_img in sample_results for src, tgt, path, cost in v]
all_invalid = [(src, tgt, path, cost, roi_img) for _, _,  iv, roi_img in sample_results for src, tgt, path, cost in iv]

print(f"\nTotal — valid: {len(all_valid)}, invalid: {len(all_invalid)}")

  S1475-2_a: valid=34, invalid=1568
  S1196-2_b: valid=20, invalid=3115
  S1196-2_a: valid=31, invalid=2107
  S1475-2_b: valid=51, invalid=2986
  S1313-2_b: valid=29, invalid=3139
  S1430-2_a: valid=20, invalid=4640
  S1585-2_a: valid=38, invalid=3823
  S1616-2_a: valid=68, invalid=5376
  S1401-2_b: valid=167, invalid=6406
  S1694-2_a: valid=47, invalid=3178
  S1430-2_b: valid=65, invalid=6528
  S1401-2_a: valid=174, invalid=5730
  S1313-2_a: valid=37, invalid=6896
  S1585-2_b: valid=56, invalid=6896
  S1734-2_b: valid=27, invalid=2740
  S1140-2_a: valid=92, invalid=9112
  S1734-2_a: valid=58, invalid=3091
  S1494-2_b: valid=172, invalid=8741
  S1140-2_b: valid=166, invalid=9512
  S1672-2_a: valid=123, invalid=8209
  S1694-2_b: valid=119, invalid=7072
  S1616-2_b: valid=165, invalid=15109
  S1278-2_b: valid=86, invalid=12748
  S1672-2_b: valid=213, invalid=9661
  S226-2_b: valid=41, invalid=3228
  S1494-2_a: valid=123, invalid=13660
  S226-2_a: valid=29, invalid=4060
  S1571-2_b: valid

In [14]:
# Per-sample visualization: each sample = one row (valid | invalid)
valid_samples = [(name, v, iv, img) for name, v, iv, img in sample_results if img is not None]
n_samples = len(valid_samples)

# fig, axes = plt.subplots(n_samples, 2, figsize=(32, 6 * n_samples), constrained_layout=True)
# if n_samples == 1:
#     axes = axes[np.newaxis, :]

# for row, (name, v_paths, iv_paths, roi_img) in enumerate(valid_samples):
#     for col, (path_list, color, label) in enumerate([
#         (v_paths,  (0, 220, 0),   'Valid'),
#         (iv_paths, (220, 60, 60), 'Invalid'),
#     ]):
#         viz = cv2.cvtColor(roi_img, cv2.COLOR_GRAY2RGB)
#         for src, tgt, path, cost in path_list:
#             for i in range(len(path) - 1):
#                 y0, x0 = path[i]
#                 y1, x1 = path[i + 1]
#                 cv2.line(viz, (x0, y0), (x1, y1), color, 1)
#         axes[row, col].imshow(viz)
#         axes[row, col].set_title(f'{name} — {label} (n={len(path_list)})', fontsize=9)
#         axes[row, col].axis('off')

# plt.suptitle('Valid (green) vs Invalid (red) Paths — All Samples', fontsize=14)
# plt.show()

In [15]:
N_POINTS = 100

def get_cost_curve(path, cost_map):
    pts = np.array(path, dtype=float)
    costs = cost_map[pts[:, 0].astype(int), pts[:, 1].astype(int)]
    diffs = np.diff(pts, axis=0)
    seg_len = np.linalg.norm(diffs, axis=1)
    arc = np.concatenate([[0.0], np.cumsum(seg_len)])
    arc_norm = arc / arc[-1]
    return arc_norm, costs

def build_curves(path_list, n=N_POINTS):
    x = np.linspace(0, 1, n)
    curves = []
    for src, tgt, path, cost, roi_img in path_list:
        if len(path) < 2:
            continue
        c_map = ((255 - roi_img.astype(np.float64)) / 255) ** 2
        arc_norm, costs = get_cost_curve(path, c_map)
        if len(np.unique(arc_norm)) < 2:
            continue
        fn = interp1d(arc_norm, costs, kind='linear', fill_value='extrapolate')
        curves.append(fn(x))
    return np.array(curves)

x_uniform = np.linspace(0, 1, N_POINTS)
valid_curves   = build_curves(all_valid)
invalid_curves = build_curves(all_invalid)

print(f"Valid curves:   {len(valid_curves)}")
print(f"Invalid curves: {len(invalid_curves)}")

def plot_curves(ax, curves, color, label):
    if len(curves) == 0:
        ax.set_title(f'{label} — no data')
        return
    for c in curves:
        ax.plot(x_uniform, c, color=color, alpha=0.02, linewidth=0.6)
    p25 = np.percentile(curves, 25, axis=0)
    p75 = np.percentile(curves, 75, axis=0)
    ax.fill_between(x_uniform, p25, p75, alpha=0.25, color=color)
    ax.plot(x_uniform, curves.mean(axis=0), color=color, linewidth=2,
            label=f'{label} (n={len(curves)})')

# ── Figure 1: line plots ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5), constrained_layout=True)

plot_curves(axes[0], valid_curves,   'steelblue', 'Valid')
axes[0].set_title('Valid Paths')
axes[0].set_xlabel('Normalized Position (0=start, 1=end)')
axes[0].set_ylabel('Cost Map Value')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

plot_curves(axes[1], invalid_curves, 'tomato', 'Invalid')
axes[1].set_title('Invalid Paths')
axes[1].set_xlabel('Normalized Position (0=start, 1=end)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plot_curves(axes[2], valid_curves,   'steelblue', 'Valid')
plot_curves(axes[2], invalid_curves, 'tomato',    'Invalid')
axes[2].set_title('Valid vs Invalid — Overlaid')
axes[2].set_xlabel('Normalized Position (0=start, 1=end)')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Cost Curves Along A* Paths — All {len(sample_dirs)} Samples', fontsize=13)
plt.show()

# ── Figure 2: heatmaps ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
for ax, curves, title in [
    (axes[0], valid_curves,   f'Valid (n={len(valid_curves)})'),
    (axes[1], invalid_curves, f'Invalid (n={len(invalid_curves)})'),
]:
    if len(curves) == 0:
        ax.set_title(f'{title} — no data'); continue
    sorted_c = curves[np.argsort(curves.mean(axis=1))]
    im = ax.imshow(sorted_c, aspect='auto', origin='lower',
                   extent=[0, 1, 0, len(sorted_c)],
                   cmap='hot', interpolation='nearest')
    plt.colorbar(im, ax=ax, label='Cost')
    ax.set_xlabel('Normalized Position (0=start, 1=end)')
    ax.set_ylabel('Path Index (sorted by mean cost)')
    ax.set_title(title)

plt.suptitle('Cost Heatmaps — All Samples (sorted by mean cost)', fontsize=13)
plt.show()

KeyboardInterrupt: 